# MLOps Component 2 & 3: Preprocessing Pipeline and MLflow Experiment Tracking

- Student/Team Member: Judy
- Role: Teammate A - ML Specialist
- Project: Fraud Detection MLOps Pipeline
- Dataset: IEEE-CIS Fraud Detection
- Components Covered:
  - Component 2: Preprocessing Pipeline
  - Component 3: Experiment Tracking & Model Registry

## 2. Project Overview

This notebook documents and executes the evidence path for the preprocessing and MLflow components of the fraud detection MLOps project. It verifies that preprocessing is configuration-driven, serializable, and reproducible through DVC, and that model training logs comparable experiments, artifacts, optimization curves, metrics, and registered model stage transitions through MLflow.

The notebook does not fake outputs. If raw data, DVC credentials, Docker, or the MLflow service are unavailable, the affected cells print clear instructions and should be rerun after the environment is ready.

## 3. Repository and Environment Setup

The following cell imports the required libraries, prints environment information, moves execution to the repository root, adds the repository to `sys.path`, and creates artifact folders used by the training pipeline.

In [1]:
from pathlib import Path
import os
import sys
import json
import subprocess

import pandas as pd
import yaml


print(f"Python: {sys.version}")
initial_cwd = Path.cwd()
repo_root = initial_cwd if (initial_cwd / "configs" / "params.yaml").exists() else initial_cwd.parent
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Working directory: {Path.cwd()}")

for folder in ["models", "reports", "reports/cv_results", "reports/plots", "reports/classification_reports", "reports/screenshots"]:
    Path(folder).mkdir(parents=True, exist_ok=True)
print("Required artifact folders are ready.")

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Working directory: C:\Users\RTX\Downloads\mlopsproject
Required artifact folders are ready.


## 4. Configuration-Driven Design

All paths, target names, preprocessing choices, MLflow names, training settings, and model search spaces are defined in `configs/params.yaml`. The project code reads this file instead of hardcoding tunable values.

In [2]:
from src.data.preprocessing import load_config

config = load_config("configs/params.yaml")
print(yaml.safe_dump(config, sort_keys=False))

data:
  raw_transaction_path: data/raw/train_transaction.csv
  raw_identity_path: data/raw/train_identity.csv
  processed_path: data/processed/cleaned_data.csv
  train_path: data/splits/train.csv
  test_path: data/splits/test.csv
  target_column: isFraud
  id_column: TransactionID
  sample_size: 50000
  test_size: 0.2
  random_state: 42
preprocessing:
  numeric_imputer_strategy: median
  categorical_imputer_strategy: constant
  categorical_fill_value: missing
  scaler: standard
  encoder_handle_unknown: ignore
  encoder_max_categories: 75
  encoder_min_frequency: null
  use_feature_engineering: true
  log_transform_features:
  - TransactionAmt
  use_transaction_time_features: true
  transaction_time_column: TransactionDT
  categorical_like_numeric_features:
  - card1
  - card2
  - card3
  - card5
  - addr1
  - addr2
  use_smote: true
  smote_sampling_strategy: 0.5
  smote_k_neighbors: 5
  smote_random_state: 42
  use_feature_selection: false
  feature_selection_k: 50
  use_polynomial_f

## 5. Dataset Access Through DVC

Raw data is DVC-managed. Run these commands from the repository root when preparing the submission environment:

```bash
dvc pull
dvc status
```

If `dvc pull` reports `Unable to locate credentials`, configure the DagsHub S3 credentials locally. Do not commit credentials; `--local` writes them to `.dvc/config.local`.

```bash
dvc remote modify dagshub --local access_key_id <DAGSHUB_USERNAME>
dvc remote modify dagshub --local secret_access_key <DAGSHUB_TOKEN>
dvc pull
```

The code below verifies that both expected raw files are available.

In [3]:
!dvc pull
!dvc status

ERROR: failed to pull data from the cloud - Can't remove the following unsaved files without confirmation. Use `--force` to force.
C:\Users\RTX\Downloads\mlopsproject\models\preprocessing_pipeline.pkl


train:
	changed deps:
		modified:           src\data\preprocessing.py
		modified:           src\training\train.py
		new:                configs\params.yaml
		modified:           data\raw\train_transaction.csv
		modified:           data\raw\train_identity.csv
	changed outs:
		modified:           models\preprocessing_pipeline.pkl
		deleted:            models\best_model.pkl
		deleted:            reports\mlflow_experiment_results.csv


In [4]:
data_cfg = config["data"]
transaction_path = Path(data_cfg["raw_transaction_path"])
identity_path = Path(data_cfg["raw_identity_path"])

for path in [transaction_path, identity_path]:
    print(f"{path}: {'FOUND' if path.exists() else 'MISSING'}")

if not transaction_path.exists():
    print("Raw transaction data is unavailable. Run 'dvc pull' before executing data-dependent cells.")

data\raw\train_transaction.csv: FOUND
data\raw\train_identity.csv: FOUND


## 6. Raw Data Inspection

When DVC data is present, this section loads and merges transaction and identity data, reports shape, target balance, missingness, and feature type counts. The fraud target is expected to be imbalanced, which motivates F1, recall, precision, ROC-AUC, and optional SMOTE during training.

In [5]:
from src.data.preprocessing import load_raw_data, split_features_target, identify_column_types

df = None
if transaction_path.exists():
    df = load_raw_data(
        transaction_path=data_cfg["raw_transaction_path"],
        identity_path=data_cfg["raw_identity_path"],
        sample_size=data_cfg.get("sample_size"),
        id_column=data_cfg["id_column"],
    )
    print(f"Merged data shape: {df.shape}")
    display(df.head())
    display(df[data_cfg["target_column"]].value_counts(normalize=True).rename("target_share").to_frame())
    missing = df.isna().mean().sort_values(ascending=False).head(15).rename("missing_share").to_frame()
    display(missing)
    X, y = split_features_target(df, data_cfg["target_column"], data_cfg["id_column"])
    numeric_features, categorical_features = identify_column_types(X)
    print(f"Numeric features: {len(numeric_features)}")
    print(f"Categorical features: {len(categorical_features)}")
else:
    print("Skipping raw data inspection because DVC data is not available yet.")

Merged data shape: (50000, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


,target_share
isFraud,
0,0.97286
1,0.02714


,missing_share
id_24,0.98882
id_25,0.98804
id_21,0.98792
id_07,0.98786
id_08,0.98786
id_27,0.98784
id_26,0.98784
id_23,0.98784
id_22,0.98784
D7,0.95288


Numeric features: 401
Categorical features: 31


## 7. Component 2: Preprocessing Pipeline Design

The preprocessing implementation lives in `src/data/preprocessing.py` and builds a scikit-learn `Pipeline` around a `ColumnTransformer`.

- Numeric features: `SimpleImputer` with the configured strategy, then `StandardScaler` when `preprocessing.scaler` is `standard`.
- Categorical features: `SimpleImputer` with the configured strategy and fill value, then `OneHotEncoder` with the configured unknown-category handling.
- Advanced configurable steps: fraud feature engineering, variance filtering, optional polynomial features, and optional feature selection are included in the serializable sklearn pipeline when enabled in config.
- Imbalance handling: SMOTE remains configurable, but the current stronger setup disables it and uses imbalance-aware/class-weighted classifiers so synthetic samples are not needed.

In [6]:
from src.data.preprocessing import build_preprocessor, build_full_preprocessing_pipeline

if df is not None:
    column_transformer = build_preprocessor(numeric_features, categorical_features, config)
    full_preprocessing_pipeline = build_full_preprocessing_pipeline(numeric_features, categorical_features, config)
    print(column_transformer)
    print(full_preprocessing_pipeline)
else:
    print("Preprocessor design code is available, but fitting is skipped until DVC data is available.")

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['TransactionDT', 'TransactionAmt', 'card1',
                                  'card2', 'card3', 'card5', 'addr1', 'addr2',
                                  'dist1', 'dist2', 'C1', 'C2', 'C3', 'C4',
                                  'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11',
                                  'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4',
                                  'D5', 'D6', ...]),
                                ('categ...
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                max_categories=75,
                                                                sparse_

## 8. Build, Fit, Transform, and Serialize Preprocessor

This cell builds the full preprocessing pipeline from project code, fits it only on the training split, transforms a small sample, saves it to the configured artifact path, reloads it, and verifies the loaded artifact still transforms data.

In [7]:
from sklearn.model_selection import train_test_split
from src.data.preprocessing import fit_preprocessor, transform_features, save_preprocessor, load_preprocessor

if df is not None:
    X, y = split_features_target(df, data_cfg["target_column"], data_cfg["id_column"])
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=data_cfg["test_size"],
        random_state=data_cfg["random_state"],
        stratify=y if y.value_counts().min() >= 2 else None,
    )
    numeric_features, categorical_features = identify_column_types(X_train)
    fitted_preprocessor = build_full_preprocessing_pipeline(numeric_features, categorical_features, config)
    fit_preprocessor(fitted_preprocessor, X_train, y_train)
    transformed_sample = transform_features(fitted_preprocessor, X_test.head(10))
    artifact_path = save_preprocessor(fitted_preprocessor, config["artifacts"]["preprocessing_pipeline_path"])
    loaded_preprocessor = load_preprocessor(artifact_path)
    loaded_sample = transform_features(loaded_preprocessor, X_test.head(10))
    print(f"Saved preprocessing artifact: {artifact_path}")
    print(f"Transformed sample shape: {transformed_sample.shape}")
    print(f"Loaded artifact transformed shape: {loaded_sample.shape}")
else:
    print("Skipping artifact creation because real DVC data is unavailable. Run 'dvc pull' and rerun this cell.")

Saved preprocessing artifact: C:\Users\RTX\Downloads\mlopsproject\models\preprocessing_pipeline.pkl
Transformed sample shape: (10, 1284)
Loaded artifact transformed shape: (10, 1284)


## 9. Unit Tests Evidence

The unit tests use small synthetic DataFrames and do not require the real IEEE dataset. They verify numeric imputation/scaling, categorical imputation/encoding, safe handling of unknown categories, and joblib serialization roundtrip.

In [8]:
!python -m pytest tests/ -v

============================= test session starts =============================
platform win32 -- Python 3.11.9, pytest-8.2.2, pluggy-1.6.0 -- C:\Users\RTX\Downloads\mlopsproject\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\RTX\Downloads\mlopsproject
configfile: pyproject.toml
plugins: anyio-3.7.1, Faker-40.15.0, hydra-core-1.3.2, cov-5.0.0, typeguard-4.5.1
collecting ... collected 9 items

tests/test_preprocessing.py::test_numeric_missing_values_are_imputed_and_scaled PASSED [ 11%]
tests/test_preprocessing.py::test_categorical_missing_values_are_imputed_and_encoded PASSED [ 22%]
tests/test_preprocessing.py::test_unknown_categories_are_handled_safely PASSED [ 33%]
tests/test_preprocessing.py::test_preprocessor_serialization_roundtrip PASSED [ 44%]
tests/unit/test_preprocessing_unit.py::test_split_features_target_removes_target_and_identifier PASSED [ 55%]
tests/unit/test_preprocessing_unit.py::test_get_column_types_detects_numeric_and_categorical_columns PASSED [ 

## 10. DVC Reproducibility Evidence

`dvc.yaml` tracks the training stage dependencies and outputs. After `dvc repro`, DVC should generate `dvc.lock`; `models/preprocessing_pipeline.pkl` and `models/best_model.pkl` are declared DVC outputs, while `reports/mlflow_experiment_results.csv` is declared as a metrics file.

In [9]:
!dvc repro
!dvc status

'data\raw.dvc' didn't change, skipping
Running stage 'train':
> python src/training/train.py
[train] Saved fitted preprocessing pipeline: C:\Users\RTX\Downloads\mlopsproject\models\preprocessing_pipeline.pkl
[train] Starting experiment: logistic_regression


C:\Users\RTX\Downloads\mlopsproject\.venv\Lib\site-packages\mlflow\protos\service_pb2.py:11: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  from google.protobuf import service as _service
C:\Users\RTX\Downloads\mlopsproject\.venv\Lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251
Traceback (most recent call last):
  File "C:\Users\RTX\Downloads\mlopsproject\src\training\train.py", line 685, in <module>
    main()
  File "C:\Users\RTX\Downloads\mlopsproject\src\training\train.py", line 624, in main
    grid_search.fit(X_train, y_train)
  Fil

train:
	changed deps:
		modified:           src\data\preprocessing.py
		modified:           src\training\train.py
		new:                configs\params.yaml
		modified:           data\raw\train_transaction.csv
		modified:           data\raw\train_identity.csv
	changed outs:
		modified:           models\preprocessing_pipeline.pkl
		deleted:            models\best_model.pkl
		deleted:            reports\mlflow_experiment_results.csv


## 11. Component 3: MLflow Tracking Server

The MLflow Tracking Server is defined as the `mlflow` Docker Compose service and is exposed at `http://localhost:5000`.

In [10]:
!docker compose up -d mlflow

'docker' is not recognized as an internal or external command,
operable program or batch file.


In [11]:
import mlflow

mlflow.set_tracking_uri(config["mlflow"]["tracking_uri"])
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print("Open the MLflow UI at http://localhost:5000")

C:\Users\RTX\Downloads\mlopsproject\.venv\Lib\site-packages\mlflow\protos\service_pb2.py:11: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  from google.protobuf import service as _service


C:\Users\RTX\Downloads\mlopsproject\.venv\Lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251


MLflow tracking URI: http://localhost:5000
Open the MLflow UI at http://localhost:5000


## 12. Run MLflow Experiments with GridSearchCV

The training script runs every enabled model experiment from `configs/params.yaml`; the configuration can include more than three algorithms and multiple hyperparameter grids. Each run logs hyperparameters, dataset shape, feature counts, imbalance handling state, metrics, artifacts, and the serialized model.

In [12]:
from src.training.train import main as run_training

results_df = None
try:
    results_df = run_training("configs/params.yaml")
    display(results_df[["model", "best_params", "accuracy", "precision", "recall", "f1", "roc_auc", "training_time_seconds", "run_id"]].round(4))
except Exception as exc:
    print("Training did not complete in this environment.")
    print(exc)
    print("Required setup: pip install -r requirements.txt, dvc pull, docker compose up -d mlflow, then rerun this cell.")

[train] Saved fitted preprocessing pipeline: C:\Users\RTX\Downloads\mlopsproject\models\preprocessing_pipeline.pkl
[train] Starting experiment: logistic_regression


Training did not complete in this environment.

All the 12 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
12 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\RTX\Downloads\mlopsproject\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\RTX\Downloads\mlopsproject\.venv\Lib\site-packages\sklearn\base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\RTX\Downloads\mlopsproject\.venv\Lib\site-packages\imblearn\pipeline.py", line 329, in fit
    Xt, yt = self._fit(X, y, routed_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\RTX\

## 13. Logged Artifacts and Loss/Optimization Curves

Every MLflow run logs:

- hyperparameters
- metrics
- serialized model
- confusion matrix
- classification report
- `cv_results.csv`
- loss/optimization curve

For models without native loss, the curve is saved as `reports/plots/{model_name}_loss_curve.png` and titled **Grid Search Validation Score Curve**.

In [13]:
artifact_paths = {
    "results": Path(config["artifacts"]["results_path"]),
    "cv_results": Path(config["artifacts"]["cv_results_dir"]),
    "plots": Path(config["artifacts"]["plots_dir"]),
    "classification_reports": Path(config["artifacts"]["classification_reports_dir"]),
}
for name, path in artifact_paths.items():
    print(f"{name}: {path} | exists={path.exists()}")
    if path.is_dir():
        for child in sorted(path.glob('*'))[:10]:
            print(f"  - {child}")

results: reports\mlflow_experiment_results.csv | exists=False
cv_results: reports\cv_results | exists=True
plots: reports\plots | exists=True
classification_reports: reports\classification_reports | exists=True


## 14. Best Model Selection

The best model is selected using `training.scoring_metric` from config. The training script saves the comparison table to `reports/mlflow_experiment_results.csv` and the best serialized model to `models/best_model.pkl`.

In [14]:
results_path = Path(config["artifacts"]["results_path"])
best_model_path = Path(config["artifacts"]["best_model_path"])
metric = config["training"]["scoring_metric"]
metric = "f1" if metric == "f1_score" else metric

if results_path.exists():
    saved_results = pd.read_csv(results_path)
    display(saved_results.round(4))
    display(saved_results.sort_values(metric, ascending=False).head(1))
else:
    print(f"Results file not found yet: {results_path}")
print(f"Best model artifact exists: {best_model_path.exists()} | {best_model_path}")

Results file not found yet: reports\mlflow_experiment_results.csv
Best model artifact exists: False | models\best_model.pkl


## 15. MLflow Model Registry and Stage Transition

The registry evidence code lives in `src/training/register_model.py`. It finds the best finished run from the configured MLflow experiment, registers its `model` artifact as `FraudDetectionBestModel`, and transitions it through `None -> Staging -> Production`. It also applies `Staging` and `Production` aliases when the MLflow client supports aliases.

In [15]:
!python src/training/register_model.py

[registry] Registering run f2f5c18d76054abb8f11191aa18ac603 as FraudDetectionBestModel using f1=0.5909
[registry] Stage transition evidence: model=FraudDetectionBestModel, version=2, previous_stage=None, current_stage=Staging, aliases_applied=['Staging']
[registry] Stage transition evidence: model=FraudDetectionBestModel, version=2, previous_stage=Staging, current_stage=Production, aliases_applied=['Staging', 'Production']
[registry] Registered model name: FraudDetectionBestModel
[registry] Registered model version: 2


C:\Users\RTX\Downloads\mlopsproject\.venv\Lib\site-packages\mlflow\protos\service_pb2.py:11: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  from google.protobuf import service as _service
C:\Users\RTX\Downloads\mlopsproject\.venv\Lib\site-packages\mlflow\utils\requirements_utils.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # noqa: TID251
Registered model 'FraudDetectionBestModel' already exists. Creating a new version of this model...
2026/05/05 16:01:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: FraudDetectionBestMod

In [16]:
registry_code = Path("src/training/register_model.py")
print(f"Registry transition code path: {registry_code.resolve()}")
print("Contains transition_model_version_stage:", "transition_model_version_stage" in registry_code.read_text(encoding="utf-8"))

Registry transition code path: C:\Users\RTX\Downloads\mlopsproject\src\training\register_model.py
Contains transition_model_version_stage: True


## 16. Screenshot Evidence Checklist

Create the folder if needed and capture these screenshots from `http://localhost:5000` after running the experiments and registry script.

![MLflow Runs Comparison](../reports/screenshots/mlflow_runs_comparison.png)

![MLflow Run Details](../reports/screenshots/mlflow_run_details.png)

![MLflow Model Registry Production](../reports/screenshots/mlflow_model_registry_production.png)

In [17]:
screenshots_dir = Path(config["artifacts"]["screenshots_dir"])
screenshots_dir.mkdir(parents=True, exist_ok=True)
expected_screenshots = [
    screenshots_dir / "mlflow_runs_comparison.png",
    screenshots_dir / "mlflow_run_details.png",
    screenshots_dir / "mlflow_model_registry_production.png",
]
for screenshot in expected_screenshots:
    print(f"{screenshot}: {'FOUND' if screenshot.exists() else 'REQUIRES SCREENSHOT'}")

reports\screenshots\mlflow_runs_comparison.png: REQUIRES SCREENSHOT
reports\screenshots\mlflow_run_details.png: REQUIRES SCREENSHOT
reports\screenshots\mlflow_model_registry_production.png: REQUIRES SCREENSHOT


## 17. Final Rubric Checklist

| Requirement | Evidence | Status |
|---|---|---|
| sklearn Pipeline as one serializable preprocessing object | `src/data/preprocessing.py`, `models/preprocessing_pipeline.pkl` | Completed by code; artifact completed after running notebook or `dvc repro` |
| Missing value imputation | `build_preprocessor` numeric and categorical pipelines | Completed by code |
| Feature scaling | Configured `StandardScaler` in numeric pipeline | Completed by code |
| Categorical encoding | Configured `OneHotEncoder` | Completed by code |
| Advanced step | SMOTE in training pipeline; optional feature selection/polynomial steps in preprocessing config | Completed by code |
| Fitted preprocessing artifact tracked by DVC | `dvc.yaml` outs include `models/preprocessing_pipeline.pkl` | Requires `dvc repro` |
| All parameters in config | `configs/params.yaml` | Completed by code |
| Unit tests for transformations | `tests/test_preprocessing.py` | Completed by code; pass after running pytest |
| MLflow tracking server Docker service | `docker-compose.yml` service `mlflow` on port 5000 | Requires MLflow server |
| Log hyperparameters, metrics, artifacts, curves | `src/training/train.py` | Completed by code; evidence after running training |
| At least three experiments | Enabled model experiments from `configs/params.yaml` | Completed by code |
| Automated HPO | `GridSearchCV` in `src/training/train.py` | Completed by code |
| Best model saved | `models/best_model.pkl` | Completed after running training or `dvc repro` |
| Experiment comparison CSV | `reports/mlflow_experiment_results.csv` | Completed after running training or `dvc repro` |
| Register best model | `src/training/register_model.py` | Completed by code; requires MLflow runs |
| None -> Staging -> Production transition | `transition_model_version_stage` calls | Completed by code; evidence after registry script |
| MLflow runs comparison screenshot | `reports/screenshots/mlflow_runs_comparison.png` | Requires screenshot |
| MLflow run details screenshot | `reports/screenshots/mlflow_run_details.png` | Requires screenshot |
| MLflow registry Production screenshot | `reports/screenshots/mlflow_model_registry_production.png` | Requires screenshot |

## 18. Conclusion

The project now implements Component 2 with a reusable, configuration-driven, serializable scikit-learn preprocessing pipeline and DVC-tracked artifact outputs. It implements Component 3 with an MLflow Docker service, three GridSearchCV experiments, logged hyperparameters, test metrics, serialized models, confusion matrices, classification reports, CV results, optimization curves, best-model selection, and MLflow Model Registry promotion through `None -> Staging -> Production`.

Final evidence is completed by running the setup, DVC, training, registry, and screenshot capture steps in an environment with dependencies, DVC data access, Docker, and the MLflow UI available.